In [2]:
# 03_tempdiff_clustering.py  (or use in a fresh notebook cell)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# --------------------------
# CONFIG
# --------------------------
DATA_DIR = os.path.join("..", "data_all")
SESS_FILE = os.path.join(DATA_DIR, "SeccSessionStop_last_year.csv")

OUTPUT_DIR = "../tempdiff_clustering_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

USE_SILHOUETTE_SWEEP = False   # set True to auto-pick k (3..6); False uses K=4
K_DEFAULT = 4

MIN_WEEKS = 12                 # outlets must have ≥ this many weekly points
RECENT_WEEKS_FOR_RISK = 8      # slope window to identify "rising" cluster
SIX_MONTH_WEEKS = 26           # ~6 months at weekly granularity

# --------------------------
# HELPERS
# --------------------------
def split_idoutlet(id_str):
    if pd.isna(id_str):
        return (pd.NA, pd.NA)
    s = str(id_str)
    if s and s[-1].isdigit():
        return (s[:-1], int(s[-1]))
    return (s, pd.NA)

def weekly_tempdiff_matrix(sess_df):
    # Timestamps + weekly bucket
    sess_df["@timestamp"] = pd.to_datetime(sess_df["@timestamp"], errors="coerce")
    sess_df = sess_df.sort_values("@timestamp")
    sess_df["week"] = sess_df["@timestamp"].dt.to_period("W-MON").dt.start_time

    # Aggregate: weekly mean of temp diff
    # Your column is 'diff' for temp difference in SeccSessionStop
    wk = (sess_df
          .groupby(["@logStream", "outlet", "week"], as_index=False)["diff"]
          .mean()
          .rename(columns={"diff": "tempdiff"}))

    # Pivot to matrix: rows = outlet pairs, cols = weeks
    mat = wk.pivot_table(index=["@logStream", "outlet"], columns="week", values="tempdiff")
    # Keep only outlets with enough weeks
    mat = mat.loc[mat.notna().sum(axis=1) >= MIN_WEEKS]
    return mat

def interpolate_and_standardize(mat):
    # Interpolate across time (columns), then ff ill/bfill edges
    mat_interp = mat.sort_index(axis=1).copy()
    mat_interp = mat_interp.interpolate(axis=1, limit_direction="both")
    mat_interp = mat_interp.fillna(method="ffill", axis=1).fillna(method="bfill", axis=1)

    # Row-wise z-score (shape-based)
    # (value - row_mean) / row_std, safe when std=0 -> set to 0
    row_means = mat_interp.mean(axis=1)
    row_stds  = mat_interp.std(axis=1).replace(0, np.nan)
    z = (mat_interp.sub(row_means, axis=0)).div(row_stds, axis=0)
    z = z.fillna(0.0)
    return z

def choose_k_by_silhouette(Z, ks=(3,4,5,6), random_state=42):
    best_k, best_score = None, -1
    for k in ks:
        km = KMeans(n_clusters=k, random_state=random_state, n_init="auto").fit(Z)
        labels = km.labels_
        # Silhouette on the normalized vectors
        score = silhouette_score(Z, labels)
        if score > best_score:
            best_k, best_score = k, score
    return best_k

def compute_trend_features(row_series):
    """Return slopes & rises for ranking & cluster selection."""
    y = row_series.values.astype(float)
    # Entire period slope via simple linear fit on index
    x = np.arange(len(y))
    if len(y) >= 2:
        slope_all = np.polyfit(x, y, 1)[0]
    else:
        slope_all = 0.0

    # Recent slope (last RECENT_WEEKS_FOR_RISK)
    if len(y) >= RECENT_WEEKS_FOR_RISK:
        xr = np.arange(RECENT_WEEKS_FOR_RISK)
        yr = y[-RECENT_WEEKS_FOR_RISK:]
        slope_recent = np.polyfit(xr, yr, 1)[0]
    else:
        slope_recent = 0.0

    # 6-month (26 weeks) relative rise: (last mean - first mean) / (abs(first mean)+1e-6)
    if len(y) >= SIX_MONTH_WEEKS:
        first_mean = np.mean(y[:SIX_MONTH_WEEKS//2])
        last_mean  = np.mean(y[-SIX_MONTH_WEEKS//2:])
        rise_6m = (last_mean - first_mean) / (abs(first_mean) + 1e-6)
    else:
        rise_6m = 0.0

    return slope_all, slope_recent, rise_6m

def summarize_clusters(Z, labels, week_cols):
    """Return per-cluster avg profile and cluster-level 'recent slope' to pick a risk cluster."""
    cluster_profiles = {}
    cluster_recent_slopes = {}
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        avg_profile = Z.iloc[idx].mean(axis=0)
        cluster_profiles[c] = avg_profile
        # compute recent slope on cluster avg
        y = avg_profile.values
        if len(y) >= RECENT_WEEKS_FOR_RISK:
            xr = np.arange(RECENT_WEEKS_FOR_RISK)
            yr = y[-RECENT_WEEKS_FOR_RISK:]
            slope_recent = np.polyfit(xr, yr, 1)[0]
        else:
            slope_recent = 0.0
        cluster_recent_slopes[c] = slope_recent
    # the “risk cluster” = largest recent slope
    risk_cluster = max(cluster_recent_slopes, key=cluster_recent_slopes.get)
    return cluster_profiles, cluster_recent_slopes, risk_cluster

def plot_cluster_profiles(Z, labels, out_path):
    """Plot average curve + IQR band per cluster (z-scored)."""
    fig, ax = plt.subplots(figsize=(14, 6))
    for c in sorted(np.unique(labels)):
        members = Z[labels == c]
        mean_profile = members.mean(axis=0).values
        q1 = members.quantile(0.25, axis=0).values
        q3 = members.quantile(0.75, axis=0).values
        ax.plot(Z.columns, mean_profile, label=f"Cluster {c}")
        ax.fill_between(Z.columns, q1, q3, alpha=0.15)
    ax.set_title("Cluster average temp-diff profiles (z-scored)")
    ax.set_xlabel("Week")
    ax.set_ylabel("z-score (Temp Diff)")
    ax.legend()
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close(fig)

def plot_top_risers_in_cluster(Z, labels, risk_cluster, out_path, top_n=12):
    """Plot top-N recent risers inside the risk cluster."""
    members = Z[labels == risk_cluster]
    # rank by recent slope
    scores = []
    for i, (_, row) in enumerate(members.iterrows()):
        slope_all, slope_recent, rise_6m = compute_trend_features(row)
        scores.append((i, slope_recent))
    scores.sort(key=lambda x: x[1], reverse=True)
    sel_idx = [members.index[i] for i, _ in scores[:top_n]]

    fig, ax = plt.subplots(figsize=(14, 6))
    for idx in sel_idx:
        ax.plot(Z.columns, Z.loc[idx].values, alpha=0.6)
    ax.set_title(f"Top {top_n} recent risers (Cluster {risk_cluster})")
    ax.set_xlabel("Week")
    ax.set_ylabel("z-score (Temp Diff)")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close(fig)

# --------------------------
# MAIN
# --------------------------
# 1) Load sessions
sess_df = pd.read_csv(SESS_FILE)

# 2) Normalize IDs
sess_split = sess_df["IDOutlet"].apply(split_idoutlet)
sess_df["@logStream"] = sess_split.apply(lambda x: x[0])
sess_df["outlet"] = sess_split.apply(lambda x: x[1]).astype("Int64")

# 3) Build weekly temp-diff matrix
mat = weekly_tempdiff_matrix(sess_df)

if mat.empty:
    print("No outlets with sufficient weekly data (check MIN_WEEKS and input data).")
else:
    # 4) Interpolate & z-score rows
    Z = interpolate_and_standardize(mat)

    # 5) Choose K
    if USE_SILHOUETTE_SWEEP:
        best_k = choose_k_by_silhouette(Z, ks=(3,4,5,6))
        print(f"Chosen K by silhouette: {best_k}")
        K = best_k
    else:
        K = K_DEFAULT
        print(f"Using fixed K={K}")

    # 6) KMeans
    km = KMeans(n_clusters=K, random_state=42, n_init="auto").fit(Z)
    labels = km.labels_
    week_cols = Z.columns

    # 7) Summaries & risk cluster
    cluster_profiles, cluster_recent_slopes, risk_cluster = summarize_clusters(Z, labels, week_cols)
    print("Cluster recent slopes:", cluster_recent_slopes)
    print("Risk cluster:", risk_cluster)

    # 8) Rank all outlets by trend features (for debugging/report)
    ranks = []
    for (charger, outlet), row in Z.iterrows():
        slope_all, slope_recent, rise_6m = compute_trend_features(row)
        ranks.append({
            "@logStream": charger,
            "outlet": outlet,
            "cluster": labels[list(Z.index).index((charger, outlet))],
            "slope_all": slope_all,
            "slope_recent": slope_recent,
            "rise_6m": rise_6m
        })
    rank_df = pd.DataFrame(ranks)
    # mark risk cluster
    rank_df["is_risk_cluster"] = (rank_df["cluster"] == risk_cluster)
    # inside the risk cluster, rank by slope_recent then rise_6m
    rank_df["risk_rank"] = (
        rank_df[rank_df["is_risk_cluster"]]
        .sort_values(["slope_recent","rise_6m"], ascending=False)
        .reset_index(drop=True)
        .reset_index()
        .rename(columns={"index":"risk_rank"})
        .set_index(["@logStream","outlet"])["risk_rank"]
        .reindex(rank_df.set_index(["@logStream","outlet"]).index)
        .values
    )

    # 9) Save assignments
    out_csv = os.path.join(OUTPUT_DIR, "clusters_tempdiff.csv")
    rank_df.to_csv(out_csv, index=False)
    print(f"Saved cluster assignments & features → {out_csv}")

    # 10) Plots
    plot_cluster_profiles(Z, labels, os.path.join(OUTPUT_DIR, "cluster_profiles.png"))
    plot_top_risers_in_cluster(Z, labels, risk_cluster,
                               os.path.join(OUTPUT_DIR, "top_risk_outlets.png"),
                               top_n=12)

    # 11) Optional: export per-outlet plots (toggle)
    EXPORT_PER_OUTLET = False
    if EXPORT_PER_OUTLET:
        per_dir = os.path.join(OUTPUT_DIR, "per_outlet_plots")
        os.makedirs(per_dir, exist_ok=True)
        for (charger, outlet), row in Z.iterrows():
            fig, ax = plt.subplots(figsize=(10,4))
            ax.plot(Z.columns, row.values, marker='o')
            ax.set_title(f"{charger} – Outlet {outlet} | Cluster {labels[list(Z.index).index((charger,outlet))]}")
            ax.set_ylabel("z-score (Temp Diff)")
            fig.autofmt_xdate()
            plt.tight_layout()
            fn = f"{charger}_Outlet{outlet}.png".replace("/", "_")
            plt.savefig(os.path.join(per_dir, fn), dpi=130)
            plt.close(fig)

    # 12) Also output a single-line pipe list for the top 20 risers in risk cluster
    top20 = (rank_df[rank_df["is_risk_cluster"]]
             .sort_values(["slope_recent","rise_6m"], ascending=False)
             .head(20))
    pipe_list = "|".join(top20["@logStream"].astype(str).tolist())
    with open(os.path.join(OUTPUT_DIR, "top20_chargers_pipe.txt"), "w") as f:
        f.write(pipe_list)
    print("Top-20 chargers (risk cluster, by recent slope):")
    print(pipe_list)


C:\Users\z0054bay\AppData\Local\Temp\ipykernel_8212\1078128844.py:176: DtypeWarning: Columns (3,7) have mixed types. Specify dtype option on import or set low_memory=False.
  sess_df = pd.read_csv(SESS_FILE)
C:\Users\z0054bay\AppData\Local\Temp\ipykernel_8212\1078128844.py:42: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  sess_df["week"] = sess_df["@timestamp"].dt.to_period("W-MON").dt.start_time
C:\Users\z0054bay\AppData\Local\Temp\ipykernel_8212\1078128844.py:61: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  mat_interp = mat_interp.fillna(method="ffill", axis=1).fillna(method="bfill", axis=1)


Using fixed K=4
Cluster recent slopes: {np.int32(0): np.float64(0.2685610588523199), np.int32(1): np.float64(-0.07581086113006555), np.int32(2): np.float64(-0.1153240752190586), np.int32(3): np.float64(-0.024780599715824435)}
Risk cluster: 0
Saved cluster assignments & features → ../tempdiff_clustering_outputs\clusters_tempdiff.csv
Top-20 chargers (risk cluster, by recent slope):
6u2B90|A7al2S|foRGhQ|jBnove|H4Jbpz|AGWy6R|VWh9Py|SiOt0o|eJeyuF|KbapdX|ZFNwm9|Hj7F98|FXkaz9|30x5N4|rsKucc|zlusYd|CzxvfO|Rvx22F|QgZyBh|y6nex3
